# 03 — Controlled baseline models

Random stratified 60/20/20 evaluation. Threshold selection is performed on validation data only. The final test set remains untouched until evaluation.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
sys.path.append(str(Path.cwd().parent / 'src'))
from data import load_dataset, FEATURES, TARGET
from splits import random_60_20_20
from models import canonical_models
from evaluation import classification_metrics, select_threshold_by_validation_f1

df = load_dataset(Path('../data/raw/iotdata.csv'))
train, validation, test = random_60_20_20(df)
X_train, y_train = train[FEATURES], train[TARGET]
X_val, y_val = validation[FEATURES], validation[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

In [ ]:
results = []
for name, model in canonical_models().items():
    model.fit(X_train, y_train)
    val_score = model.predict_proba(X_val)[:, 1]
    threshold, _ = select_threshold_by_validation_f1(y_val, val_score)
    test_score = model.predict_proba(X_test)[:, 1]
    row = {'model': name, **classification_metrics(y_test, test_score, threshold)}
    results.append(row)

pd.DataFrame(results)

## Reference interpretation

The canonical portfolio reports random-split AP values of 0.003968 (Logistic Regression), 0.009588 (HistGradientBoosting) and 0.009638 (Random Forest), with movement prevalence of ~0.119%. These figures are evidence for this audited experimental record and should not be presented as universal model rankings.